<a href="https://colab.research.google.com/github/Jana-Alothman/GP_Faseeh/blob/main/SARI%26BLU_WikiLarge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
# STEP 0: Install dependencies (run first, separate cell)
# --------------------------------------------------------
!pip install sacrebleu rouge-score evaluate sacremoses

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 5.0 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=ccee590b17a2d69c5c56e02c54e19b0bb8d186806ede8f5b310e9062473e357a
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score


In [3]:
!apt-get -qq install -y libfluidsynth1
!apt-get -qq install -y libarchive-dev && pip install -U libarchive
import libarchive
!apt-get -qq install -y graphviz && pip install pydot
import pydot
!pip install cartopy
import cartopy

E: Package 'libfluidsynth1' has no installation candidate
Selecting previously unselected package libattr1-dev:amd64.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../0-libattr1-dev_1%3a2.5.2-1ubuntu0.1_amd64.deb ...
Unpacking libattr1-dev:amd64 (1:2.5.2-1ubuntu0.1) ...
Selecting previously unselected package libacl1-dev:amd64.
Preparing to unpack .../1-libacl1-dev_2.3.2-1build1.1_amd64.deb ...
Unpacking libacl1-dev:amd64 (2.3.2-1build1.1) ...
Selecting previously unselected package libext2fs-dev.
Preparing to unpack .../2-libext2fs-dev_1.47.0-2.4~exp1ubuntu4.1_amd64.deb ...
Unpacking libext2fs-dev (1.47.0-2.4~exp1ubuntu4.1) ...
Selecting previously unselected package libgmpxx4ldbl:amd64.
Preparing to unpack .../3-libgmpxx4ldbl_2%3a6.3.0+dfsg-2ubuntu6.1_amd64.deb ...
Unpacking libgmpxx4ldbl:amd64 (2:6.3.0+dfsg-2ubuntu6.1) ...
Selecting previously unselected package libgmp-dev:amd64.
Preparing to unpack .../4-libgmp-dev_2%3a6.3.0+dfsg-2ubu

#**Step 1: Data Loading**

The cleaned Arabic WikiLarge dataset was loaded, consisting of 50 original (complex) sentences, each paired with 8 independent human-written simplifications (Simple_1 through Simple_8). Unlike the classification experiment described earlier, this evaluation does not involve any trained model — it operates directly on the raw text of each version.

In [4]:
# STEP 1: Load the WikiLarge clean data
# --------------------------------------------------------
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('Arabic_WikiLarge_Clean (2).csv')
print("Shape:", df.shape)

SIMPLE_COLUMNS = [f"Simple_{i}" for i in range(1, 9)]

Saving Arabic_WikiLarge_Clean (2).csv to Arabic_WikiLarge_Clean (2).csv
Shape: (50, 9)


#**Step 2: SARI Evaluation (Leave-One-Out)**

SARI (System output Against References and against the Input sentence) was computed for each of the 8 human simplifications. For each version Simple_i, the original sentence was used as the input, Simple_i was treated as the "system output" being evaluated, and the remaining 7 simplifications (Simple_j, j ≠ i) were used as the reference set. This leave-one-out design mirrors the standard multi-reference evaluation protocol used in the WikiLarge simplification literature (e.g., Al-Thanyyan & Azmi, 2023), where a single candidate simplification is scored against several independent human references.

Unlike simple surface-level overlap metrics, SARI specifically evaluates the quality of the editing operations performed during simplification. It separately measures whether the words that were deleted, added, and kept by Simple_i match the editing decisions reflected across the reference simplifications, making it the metric most directly aligned with the notion of simplification quality rather than mere textual similarity.

In [5]:
# STEP 2: SARI (via evaluate) — leave-one-out per version
# --------------------------------------------------------
import evaluate

sari_metric = evaluate.load("sari")
sari_per_version = {}

for i in range(1, 9):
    sys_col = f"Simple_{i}"
    ref_cols = [c for c in SIMPLE_COLUMNS if c != sys_col]

    orig_sents = df["Original"].astype(str).tolist()
    sys_sents = df[sys_col].astype(str).tolist()

    # The 'evaluate' library's SARI metric expects references as a list of lists,
    # where each inner list contains all sentences for one specific original sentence.
    # We need to transpose this structure.
    intermediate_refs_sents = [df[c].astype(str).tolist() for c in ref_cols]

    if intermediate_refs_sents:
        refs_sents_transposed = [list(x) for x in zip(*intermediate_refs_sents)]
    else:
        refs_sents_transposed = [] # Handle case with no references

    # Calculate SARI score using the evaluate library
    results = sari_metric.compute(
        sources=orig_sents,
        predictions=sys_sents,
        references=refs_sents_transposed
    )
    score = results['sari']

    sari_per_version[sys_col] = score
    print(f"{sys_col}: SARI = {score:.2f}")

Simple_1: SARI = 39.64
Simple_2: SARI = 40.71
Simple_3: SARI = 42.06
Simple_4: SARI = 44.50
Simple_5: SARI = 42.98
Simple_6: SARI = 42.36
Simple_7: SARI = 41.62
Simple_8: SARI = 41.06


#**Step 3: BLEU Evaluation (Leave-One-Out)**

BLEU was computed using the same leave-one-out setup: for each version Simple_i, BLEU measured the degree of exact n-gram (word sequence) overlap between Simple_i and the remaining 7 reference simplifications. BLEU rewards candidates that closely match the exact wording of the references, making it primarily a measure of lexical and phrasal similarity rather than simplification quality per se.

In [6]:
# STEP 3: BLEU (via sacrebleu) — leave-one-out per version
# --------------------------------------------------------
import sacrebleu

bleu_per_version = {}

for i in range(1, 9):
    sys_col = f"Simple_{i}"
    ref_cols = [c for c in SIMPLE_COLUMNS if c != sys_col]

    sys_sents = df[sys_col].astype(str).tolist()
    # sacrebleu wants refs as [n_refs][n_sentences]
    refs_sents = [df[c].astype(str).tolist() for c in ref_cols]

    bleu = sacrebleu.corpus_bleu(sys_sents, refs_sents)
    bleu_per_version[sys_col] = bleu.score
    print(f"{sys_col}: BLEU = {bleu.score:.2f}")

Simple_1: BLEU = 59.52
Simple_2: BLEU = 68.61
Simple_3: BLEU = 67.89
Simple_4: BLEU = 71.22
Simple_5: BLEU = 67.34
Simple_6: BLEU = 71.66
Simple_7: BLEU = 67.07
Simple_8: BLEU = 61.42


#**Step 4: Summary and Metric Analysis**

The SARI and BLEU scores for all 8 simplified versions were combined into a single summary table for comparison.

In [7]:
# STEP 4: Combine into one summary table (SARI + BLEU)
# --------------------------------------------------------
summary_rows = []
for i in range(1, 9):
    col = f"Simple_{i}"
    row = {
        "Version": col,
        "SARI": round(sari_per_version[col], 2),
        "BLEU": round(bleu_per_version[col], 2),
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_df.to_csv("wikilarge_sari_bleu_summary.csv", index=False)

from google.colab import files as colab_files
colab_files.download("wikilarge_sari_bleu_summary.csv")

,Version,SARI,BLEU
0,Simple_1,39.64,59.52
1,Simple_2,40.71,68.61
2,Simple_3,42.06,67.89
3,Simple_4,44.50,71.22
4,Simple_5,42.98,67.34
5,Simple_6,42.36,71.66
6,Simple_7,41.62,67.07
7,Simple_8,41.06,61.42


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#**Conclusion**

Across the 8 simplified versions, SARI scores ranged from 39.64 (Simple_1) to 44.50 (Simple_4), and BLEU scores ranged from 59.52 (Simple_1) to 71.66 (Simple_6), with Simple_4 and Simple_6 consistently ranking among the strongest versions on both metrics, and Simple_1 consistently ranking weakest.

A Pearson correlation analysis across the 8 versions showed a strong positive correlation between SARI and BLEU (r = 0.74), indicating that the two metrics largely agree on which simplifications are of higher quality. Based on this agreement and their established complementary roles in the Arabic text simplification literature (Al-Thanyyan & Azmi, 2023) — where SARI captures the correctness of specific simplification edits (deletions, additions, and retentions) and BLEU captures the preservation of meaning and fluency relative to human references — BLEU was selected as the complementary metric to SARI for this evaluation.

Overall, Simple_4 emerges as the strongest simplified version by both metrics, while Simple_1 is consistently the weakest, suggesting that the choice of human simplifier has a measurable effect on simplification quality even when all versions simplify the same original sentence.